In [ ]:
# ==============================================================================
# 🎬 STAGE 2: PURE-NEURAL VIDEO VERIFICATION (COLAB)
# ==============================================================================
import os, cv2, torch, numpy as np
import torch.nn.functional as F
from tqdm import tqdm
# --- 1. SETUP ENVIRONMENT ---
print("📡 Setting up environment...")
if not os.path.exists("DehazeFormer"):
    !git clone https://github.com/IDKiro/DehazeFormer.git
import sys
sys.path.insert(0, "DehazeFormer")
!pip install -q timm einops lpips
from models.dehazeformer import DehazeFormer

📡 Setting up environment...
Cloning into 'DehazeFormer'...
remote: Enumerating objects: 94, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 94 (delta 17), reused 11 (delta 11), pack-reused 70 (from 2)
Receiving objects: 100% (94/94), 768.88 KiB | 2.84 MiB/s, done.
Resolving deltas: 100% (43/43), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 4.0 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [ ]:
# --- 2. CONFIGURATION (CHECK THESE FILENAMES) ---
WEIGHTS_PATH = "/content/dehazeformer_real_haze_best.pth"  # Your 54MB file
VIDEO_PATH   = "/content/Hazy-4.mp4"                 # Your hazy video
OUTPUT_PATH  = "/content/STAGE_2_VERIFICATION.mp4"


In [ ]:
# --- 3. LOAD STAGE 2 MODEL ---
def load_stage2_model(weights_path):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = DehazeFormer(
        in_chans=3, out_chans=4, window_size=8,
        embed_dims=[24, 48, 96, 48, 24],
        mlp_ratios=[2., 4., 4., 2., 2.],
        depths=[12, 12, 12, 6, 6],
        num_heads=[2, 4, 6, 1, 1],
        attn_ratio=[1/4, 1/2, 3/4, 0, 0],
        conv_type=['Conv', 'Conv', 'Conv', 'Conv', 'Conv']
    )

    if os.path.exists(weights_path):
        ckpt = torch.load(weights_path, map_location='cpu')
        state = ckpt.get('state_dict', ckpt.get('model', ckpt))
        new_state = { (k[7:] if k.startswith('module.') else k): v for k, v in state.items() }
        model.load_state_dict(new_state, strict=False)
        print(f"✅ Loaded Stage 2 Weights: {weights_path}")
    else:
        raise FileNotFoundError(f"❌ Weights not found! Please upload {weights_path}")

    return model.to(device).eval(), device
# --- 4. THE PURE NEURAL ENGINE (NO FILTERS) ---
@torch.no_grad()
def dehaze_pure_neural(model, frame, device):
    h, w, _ = frame.shape
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Pad to multiple of 8
    pad_h, pad_w = (8 - h % 8) % 8, (8 - w % 8) % 8

    inp = torch.from_numpy(img_rgb / 255.0).float().permute(2, 0, 1).unsqueeze(0).to(device)
    if pad_h > 0 or pad_w > 0:
        inp = F.pad(inp, (0, pad_w, 0, pad_h), mode='reflect')

    out = model(inp)[:, :3, :h, :w].squeeze(0).permute(1, 2, 0).clamp(0, 1)
    out_np = (out.cpu().numpy() * 255).astype(np.uint8)
    return cv2.cvtColor(out_np, cv2.COLOR_RGB2BGR)
# --- 5. PROCESS VIDEO ---
def process_verification_video():
    model, device = load_stage2_model(WEIGHTS_PATH)
    cap = cv2.VideoCapture(VIDEO_PATH)

    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out_writer = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (w * 2, h))

    print(f"📹 Processing {total} frames... Mode: Pure Neural Refinement")

    for _ in tqdm(range(total)):
        ret, frame = cap.read()
        if not ret: break

        # This is where the magic happens
        clean = dehaze_pure_neural(model, frame, device)

        # Create Side-by-Side Comparison
        cv2.putText(frame, "ORIGINAL HAZY", (30, 60), cv2.FONT_HERSHEY_DUPLEX, 1.5, (0,0,255), 3)
        cv2.putText(clean, "STAGE 2 PURE NEURAL", (30, 60), cv2.FONT_HERSHEY_DUPLEX, 1.5, (0,255,0), 3)
        combined = np.hstack([frame, clean])
        out_writer.write(combined)

    cap.release()
    out_writer.release()
    print(f"\n✅ Done! Download your result here: {OUTPUT_PATH}")
process_verification_video()

✅ Loaded Stage 2 Weights: /content/dehazeformer_real_haze_best.pth
📹 Processing 451 frames... Mode: Pure Neural Refinement


100%|██████████| 451/451 [04:20<00:00,  1.73it/s]


✅ Done! Download your result here: /content/STAGE_2_VERIFICATION.mp4


In [ ]:
from google.colab import files

# This will trigger a browser download for your dehazed video
files.download('/content/STAGE_2_VERIFICATION.mp4')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>